In [24]:
#Name = Utkarsh Yadav
#Branch = CSE AIMl-6
#Admission No.: 2026B15310064

In [25]:
import pandas as pd
import numpy as np

In [26]:
from google.colab import files
u = files.upload()

Saving healthcare_readmission_synthetic.csv to healthcare_readmission_synthetic (1).csv


In [27]:
filename = next(iter(u))
df = pd.read_csv(filename)

In [28]:
print(df.shape)

(1200, 13)


In [29]:
print(df.head(10))

  encounter_id patient_id admission_date discharge_date  age age_band  \
0      E000851     P00638     2024-05-05     2024-05-07   29    18-29   
1      E000731     P00474     2024-03-06     2024-03-11   33    30-44   
2      E000033     P00060     2025-12-09     2025-12-11   22    18-29   
3      E001017     P00250     2024-11-24     2024-12-12   42    30-44   
4      E000584     P00034     2025-11-28     2025-12-05   40    30-44   
5      E000470     P00607     2025-06-13     2025-06-15   61    60-74   
6      E000950     P00181     2025-09-15     2025-09-21   46    45-59   
7      E000092     P00524     2025-09-18     2025-09-28   57    45-59   
8      E000521     P00233     2024-03-05     2024-03-14   84      75+   
9      E000793     P00470     2024-07-28     2024-08-05   33    30-44   

            service primary_diagnosis            procedure admission_type  \
0  General Medicine     Heart failure        None recorded      Emergency   
1     Endocrinology    Kidney disease  Res

In [30]:
print(df.dtypes)
print(df.columns)

encounter_id                 object
patient_id                   object
admission_date               object
discharge_date               object
age                           int64
age_band                     object
service                      object
primary_diagnosis            object
procedure                    object
admission_type               object
discharge_disposition        object
length_of_stay_days           int64
readmitted_within_30_days     int64
dtype: object
Index(['encounter_id', 'patient_id', 'admission_date', 'discharge_date', 'age',
       'age_band', 'service', 'primary_diagnosis', 'procedure',
       'admission_type', 'discharge_disposition', 'length_of_stay_days',
       'readmitted_within_30_days'],
      dtype='object')


In [31]:
print(df.duplicated().sum())
print(df.isnull().sum())

0
encounter_id                  0
patient_id                    0
admission_date                0
discharge_date                0
age                           0
age_band                      0
service                       0
primary_diagnosis            11
procedure                    32
admission_type                0
discharge_disposition        12
length_of_stay_days           0
readmitted_within_30_days     0
dtype: int64


In [32]:
df.describe()

,age,length_of_stay_days,readmitted_within_30_days
count,1200.000000,1200.000000,1200.000000
mean,54.089167,4.554167,0.078333
std,21.096653,3.101626,0.268807
min,18.000000,1.000000,0.000000
25%,33.000000,2.000000,0.000000
50%,54.000000,4.000000,0.000000
75%,73.000000,6.000000,0.000000
max,90.000000,30.000000,1.000000


In [33]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   encounter_id               1200 non-null   object
 1   patient_id                 1200 non-null   object
 2   admission_date             1200 non-null   object
 3   discharge_date             1200 non-null   object
 4   age                        1200 non-null   int64 
 5   age_band                   1200 non-null   object
 6   service                    1200 non-null   object
 7   primary_diagnosis          1189 non-null   object
 8   procedure                  1168 non-null   object
 9   admission_type             1200 non-null   object
 10  discharge_disposition      1188 non-null   object
 11  length_of_stay_days        1200 non-null   int64 
 12  readmitted_within_30_days  1200 non-null   int64 
dtypes: int64(3), object(10)
memory usage: 122.0+ KB
None


In [34]:
data = df.copy()
data['admission_date'] = pd.to_datetime(data['admission_date'])
data['discharge_date'] = pd.to_datetime(data['discharge_date'])
print(data['admission_date'].min())
print(data['admission_date'].max())

2024-01-01 00:00:00
2025-12-30 00:00:00


In [35]:
print(data['patient_id'].nunique())
print(data['encounter_id'].nunique())
print(data['service'].nunique())
print(data['primary_diagnosis'].nunique())
print(data['admission_type'].nunique())
print(data['discharge_disposition'].nunique())

545
1200
5
6
3
4


In [36]:
sup = data.groupby('service')['length_of_stay_days'].mean()
print(sup)

service
Cardiology          4.464602
Endocrinology       4.689815
General Medicine    4.551724
Respiratory         4.572816
Surgery             4.472603
Name: length_of_stay_days, dtype: float64


In [37]:
sumy = data.groupby('patient_id').agg({
    'length_of_stay_days': ['mean', 'sum'],
    'readmitted_within_30_days': 'sum',
    'encounter_id': 'count'
})
sumy.columns = ['Average_Length_of_Stay', 'Total_Length_of_Stay',
                'Total_Readmissions', 'Total_Encounters']
print(sumy.head())

            Average_Length_of_Stay  Total_Length_of_Stay  Total_Readmissions  \
patient_id                                                                     
P00003                        3.00                     3                   0   
P00004                        4.75                    19                   1   
P00005                        5.50                    22                   1   
P00006                        4.50                     9                   0   
P00007                        3.50                     7                   0   

            Total_Encounters  
patient_id                    
P00003                     1  
P00004                     4  
P00005                     4  
P00006                     2  
P00007                     2  


In [38]:
summary = data.groupby('service').agg({
    'age': 'mean',
    'length_of_stay_days': 'mean',
    'readmitted_within_30_days': ['sum', 'mean'],
    'encounter_id': 'count'
})
summary.columns = ['Average_Age', 'Average_Length_of_Stay',
                   'Total_Readmissions', 'Readmission_Rate',
                   'Total_Encounters']
summary['Readmission_Rate'] = summary['Readmission_Rate'] * 100
print(summary)

                  Average_Age  Average_Length_of_Stay  Total_Readmissions  \
service                                                                     
Cardiology          56.132743                4.464602                  19   
Endocrinology       53.356481                4.689815                  19   
General Medicine    53.766010                4.551724                  33   
Respiratory         53.689320                4.572816                  14   
Surgery             53.472603                4.472603                   9   

                  Readmission_Rate  Total_Encounters  
service                                               
Cardiology                8.407080               226  
Endocrinology             8.796296               216  
General Medicine          8.128079               406  
Respiratory               6.796117               206  
Surgery                   6.164384               146  


In [39]:
summary = summary.sort_values('Total_Readmissions', ascending=False)
summary['Readmission_Percentage'] = (
    summary['Total_Readmissions'] / summary['Total_Encounters']
) * 100
print(summary.head(10))

                  Average_Age  Average_Length_of_Stay  Total_Readmissions  \
service                                                                     
General Medicine    53.766010                4.551724                  33   
Cardiology          56.132743                4.464602                  19   
Endocrinology       53.356481                4.689815                  19   
Respiratory         53.689320                4.572816                  14   
Surgery             53.472603                4.472603                   9   

                  Readmission_Rate  Total_Encounters  Readmission_Percentage  
service                                                                       
General Medicine          8.128079               406                8.128079  
Cardiology                8.407080               226                8.407080  
Endocrinology             8.796296               216                8.796296  
Respiratory               6.796117               206             

In [40]:
summary['Risk'] = 'Normal'
summary.loc[summary['Readmission_Rate'] > summary['Readmission_Rate'].mean(), 'Risk'] = 'High'
print(summary)

                  Average_Age  Average_Length_of_Stay  Total_Readmissions  \
service                                                                     
General Medicine    53.766010                4.551724                  33   
Cardiology          56.132743                4.464602                  19   
Endocrinology       53.356481                4.689815                  19   
Respiratory         53.689320                4.572816                  14   
Surgery             53.472603                4.472603                   9   

                  Readmission_Rate  Total_Encounters  Readmission_Percentage  \
service                                                                        
General Medicine          8.128079               406                8.128079   
Cardiology                8.407080               226                8.407080   
Endocrinology             8.796296               216                8.796296   
Respiratory               6.796117               206        

In [41]:
age_summary = data.groupby('age_band').agg({
    'encounter_id': 'count',
    'readmitted_within_30_days': ['sum', 'mean'],
    'length_of_stay_days': 'mean'
})
age_summary.columns = ['Total_Encounters', 'Total_Readmissions',
                        'Readmission_Rate', 'Average_Length_of_Stay']
age_summary['Readmission_Rate'] = age_summary['Readmission_Rate'] * 100
print(age_summary)

          Total_Encounters  Total_Readmissions  Readmission_Rate  \
age_band                                                           
18-29                  196                  17          8.673469   
30-44                  242                  15          6.198347   
45-59                  262                  21          8.015267   
60-74                  224                  21          9.375000   
75+                    276                  20          7.246377   

          Average_Length_of_Stay  
age_band                          
18-29                   4.500000  
30-44                   4.768595  
45-59                   4.503817  
60-74                   4.687500  
75+                     4.344203  


In [42]:
diagnosis_summary = data.groupby('primary_diagnosis').agg({
    'encounter_id': 'count',
    'readmitted_within_30_days': ['sum', 'mean'],
    'length_of_stay_days': 'mean'
})
diagnosis_summary.columns = ['Total_Encounters', 'Total_Readmissions',
                             'Readmission_Rate', 'Average_Length_of_Stay']
diagnosis_summary['Readmission_Rate'] = diagnosis_summary['Readmission_Rate'] * 100
print(diagnosis_summary.sort_values('Readmission_Rate', ascending=False))

                   Total_Encounters  Total_Readmissions  Readmission_Rate  \
primary_diagnosis                                                           
Heart failure                   199                  21         10.552764   
COPD                            162                  13          8.024691   
Type 2 diabetes                 342                  26          7.602339   
Hypertension                    150                  11          7.333333   
Kidney disease                  142                   9          6.338028   
Pneumonia                       194                  11          5.670103   

                   Average_Length_of_Stay  
primary_diagnosis                          
Heart failure                    5.135678  
COPD                             4.135802  
Type 2 diabetes                  4.657895  
Hypertension                     4.193333  
Kidney disease                   4.478873  
Pneumonia                        4.402062  


In [43]:
admission_summary = data.groupby('admission_type').agg({
    'encounter_id': 'count',
    'readmitted_within_30_days': ['sum', 'mean']
})
admission_summary.columns = ['Total_Encounters', 'Total_Readmissions',
                             'Readmission_Rate']
admission_summary['Readmission_Rate'] = admission_summary['Readmission_Rate'] * 100
print(admission_summary)

                Total_Encounters  Total_Readmissions  Readmission_Rate
admission_type                                                        
Elective                     298                  25          8.389262
Emergency                    671                  53          7.898659
Urgent                       231                  16          6.926407


In [44]:
discharge_summary = data.groupby('discharge_disposition').agg({
    'encounter_id': 'count',
    'readmitted_within_30_days': ['sum', 'mean']
})
discharge_summary.columns = ['Total_Encounters', 'Total_Readmissions',
                             'Readmission_Rate']
discharge_summary['Readmission_Rate'] = discharge_summary['Readmission_Rate'] * 100
print(discharge_summary)

                          Total_Encounters  Total_Readmissions  \
discharge_disposition                                            
Home                                   867                  67   
Other facility                          66                   3   
Rehabilitation                         136                  10   
Skilled nursing facility               119                  14   

                          Readmission_Rate  
discharge_disposition                       
Home                              7.727797  
Other facility                    4.545455  
Rehabilitation                    7.352941  
Skilled nursing facility         11.764706  


In [45]:
data['admission_month'] = data['admission_date'].dt.to_period('M')
monthly = data.groupby('admission_month').agg({
    'encounter_id': 'count',
    'readmitted_within_30_days': ['sum', 'mean']
})
monthly.columns = ['Total_Encounters', 'Total_Readmissions',
                   'Readmission_Rate']
monthly['Readmission_Rate'] = monthly['Readmission_Rate'] * 100
print(monthly)

                 Total_Encounters  Total_Readmissions  Readmission_Rate
admission_month                                                        
2024-01                        49                   5         10.204082
2024-02                        44                   3          6.818182
2024-03                        53                   6         11.320755
2024-04                        59                   1          1.694915
2024-05                        46                   5         10.869565
2024-06                        50                   4          8.000000
2024-07                        55                   5          9.090909
2024-08                        62                   4          6.451613
2024-09                        46                   0          0.000000
2024-10                        58                   5          8.620690
2024-11                        51                   7         13.725490
2024-12                        43                   2          4

In [46]:
summary = summary.sort_values('Readmission_Rate', ascending=False)
print(summary.head(10))

                  Average_Age  Average_Length_of_Stay  Total_Readmissions  \
service                                                                     
Endocrinology       53.356481                4.689815                  19   
Cardiology          56.132743                4.464602                  19   
General Medicine    53.766010                4.551724                  33   
Respiratory         53.689320                4.572816                  14   
Surgery             53.472603                4.472603                   9   

                  Readmission_Rate  Total_Encounters  Readmission_Percentage  \
service                                                                        
Endocrinology             8.796296               216                8.796296   
Cardiology                8.407080               226                8.407080   
General Medicine          8.128079               406                8.128079   
Respiratory               6.796117               206        